# Question 1 

In [1]:
import numpy as np
from q1 import simulate_lognormal_sum, wasserstein_1d, reject_abc
from ABCMCMC import ABCMCMC

In [3]:
seed_value = 42
main_rng = np.random.default_rng(seed_value)
    
L_true = 10
mu_true = 0.0
sigma_true = 0.3
n_obs = 200  
    
print("Generating strictly defined observational data...")
Y_obs = simulate_lognormal_sum(n_obs, L_true, mu_true, sigma_true, rng=main_rng)

##### epsilon and the prior affect the result ###
    
epsilon_val = 0.5 
    
prior_configs = [
        {"s": 1.0, "t": 1.0, "name": "Well-calibrated"},
        {"s": 10.0, "t": 10.0, "name": "Highly diffuse (Efficiency drop)"},
        {"s": 0.01, "t": 0.01, "name": "Highly concentrated (Prior bias risk)"}
    ]
    
print(f"\n--- Rejection-ABC Empirical Results (Target: mu={mu_true}, sigma={sigma_true}) ---")
for config in prior_configs:
    print(f"\nEvaluating '{config['name']}' prior specification (s={config['s']}, t={config['t']}):")
        
        # Pass the isolated generator down the execution pipeline
    samples, acc_rate = reject_abc(
        Y_obs, L_true, config['s'], config['t'], epsilon_val, num_samples=50, rng=main_rng
        )
        
    print(f"  -> Acceptance Rate : {acc_rate:.4%}")
        
    if len(samples) > 0:
        mu_est = np.mean(samples[:, 0])
        sigma_est = np.mean(samples[:, 1])
        print(f"  -> Estimated mu    : {mu_est:.4f}")
        print(f"  -> Estimated sigma : {sigma_est:.4f}")
    else:
        print("  -> Estimated mu    : N/A (0 samples accepted)")
        print("  -> Estimated sigma : N/A (0 samples accepted)")

Generating strictly defined observational data...

--- Rejection-ABC Empirical Results (Target: mu=0.0, sigma=0.3) ---

Evaluating 'Well-calibrated' prior specification (s=1.0, t=1.0):
    [!] Computational limit reached (10000 iterations). Only 25 samples accepted.
  -> Acceptance Rate : 0.2500%
  -> Estimated mu    : -0.0377
  -> Estimated sigma : 0.3799

Evaluating 'Highly diffuse (Efficiency drop)' prior specification (s=10.0, t=10.0):
    [!] Computational limit reached (10000 iterations). Only 4 samples accepted.
  -> Acceptance Rate : 0.0400%
  -> Estimated mu    : -0.0422
  -> Estimated sigma : 0.3467

Evaluating 'Highly concentrated (Prior bias risk)' prior specification (s=0.01, t=0.01):
    [!] Computational limit reached (10000 iterations). Only 0 samples accepted.
  -> Acceptance Rate : 0.0000%
  -> Estimated mu    : N/A (0 samples accepted)
  -> Estimated sigma : N/A (0 samples accepted)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from q1 import simulate_lognormal_sum, wasserstein_1d, reject_abc

# ─────────────────────────────────────────────
#  GLOBAL SEED & TRUE PARAMETERS
# ─────────────────────────────────────────────


    
SEED      = 42
L_TRUE    = 10
MU_TRUE   = 0.0
SIGMA_TRUE = 0.3
N_OBS     = 200
N_SAMPLES = 200   # accepted samples per run (increase for smoother plots)

main_rng = np.random.default_rng(SEED)
Y_obs = simulate_lognormal_sum(N_OBS, L_TRUE, MU_TRUE, SIGMA_TRUE, rng=main_rng)

# ─────────────────────────────────────────────
#  STYLE
# ─────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f14",
    "axes.facecolor":   "#16161f",
    "axes.edgecolor":   "#3a3a52",
    "axes.labelcolor":  "#c8c8e0",
    "xtick.color":      "#7a7a9a",
    "ytick.color":      "#7a7a9a",
    "text.color":       "#c8c8e0",
    "grid.color":       "#2a2a3a",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
    "axes.titlesize":   10,
    "axes.labelsize":   9,
})

ACCENT   = ["#7eb8f7", "#f7a07e", "#7ef7c0"]   # blue / orange / green
TRUE_CLR = "#ff4f6e"
PRIOR_CLR = "#a07ef7"


# ─────────────────────────────────────────────
#  HELPER : confidence ellipse (1-sigma)
# ─────────────────────────────────────────────
def plot_ellipse(ax, samples, color, n_std=2.0):
    if len(samples) < 3:
        return
    cov = np.cov(samples[:, 0], samples[:, 1])
    mean = samples.mean(axis=0)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(mean, width=w, height=h, angle=angle,
                  edgecolor=color, facecolor="none", lw=1.5, linestyle="--", alpha=0.7)
    ax.add_patch(ell)


# ═══════════════════════════════════════════════════════════════
#  FIGURE 1 — Impact of prior (s, t)   [epsilon fixed]
# ═══════════════════════════════════════════════════════════════
EPSILON_FIXED = 0.5
PRIOR_CONFIGS = [
    {"s": 1.0,  "t": 1.0,  "name": "Well-calibrated\n(s=t=1)"},
    {"s": 10.0, "t": 10.0, "name": "Highly diffuse\n(s=t=10)"},
    {"s": 0.01, "t": 0.01, "name": "Highly concentrated\n(s=t=0.01)"},
]

fig1, axes1 = plt.subplots(3, 3, figsize=(14, 11))
fig1.suptitle(
    f"Reject-ABC — Impact of Prior   [ε = {EPSILON_FIXED}  |  μ* = {MU_TRUE}  σ* = {SIGMA_TRUE}]",
    fontsize=13, color="#e0e0f5", y=0.98
)

for col, cfg in enumerate(PRIOR_CONFIGS):
    rng = np.random.default_rng(SEED + col)          # isolated seed per config
    samples, acc_rate = reject_abc(
        Y_obs, L_TRUE, cfg["s"], cfg["t"],
        EPSILON_FIXED, num_samples=N_SAMPLES,
        max_attempts=200_000, rng=rng
    )

    color = ACCENT[col]
    ax_sc  = axes1[0, col]
    ax_mu  = axes1[1, col]
    ax_sig = axes1[2, col]

    # ── column title ──
    ax_sc.set_title(
        f"{cfg['name']}\nAcc. rate = {acc_rate:.3%}",
        color=color, pad=8
    )

    # ── ROW 0 : scatter (mu, sigma) ──
    if len(samples) > 0:
        ax_sc.scatter(samples[:, 0], samples[:, 1],
                      alpha=0.35, s=12, color=color, rasterized=True)
        plot_ellipse(ax_sc, samples, color)
    ax_sc.axvline(MU_TRUE,    color=TRUE_CLR, lw=1.4, linestyle="-",  label=f"μ*={MU_TRUE}")
    ax_sc.axhline(SIGMA_TRUE, color=TRUE_CLR, lw=1.4, linestyle="--", label=f"σ*={SIGMA_TRUE}")
    ax_sc.set_xlabel("μ")
    ax_sc.set_ylabel("σ")
    ax_sc.legend(fontsize=7, loc="upper right")
    ax_sc.grid(True)

    # ── ROW 1 : histogram of mu ──
    if len(samples) > 0:
        ax_mu.hist(samples[:, 0], bins=25, density=True,
                   color=color, alpha=0.7, edgecolor="#0f0f14")
    # overlay prior
    x_range = np.linspace(-4 * cfg["s"], 4 * cfg["s"], 300)
    prior_pdf = (1 / (cfg["s"] * np.sqrt(2 * np.pi))) * np.exp(-0.5 * (x_range / cfg["s"]) ** 2)
    ax_mu.plot(x_range, prior_pdf, color=PRIOR_CLR, lw=1.5, alpha=0.8, label="Prior π(μ)")
    ax_mu.axvline(MU_TRUE, color=TRUE_CLR, lw=1.5, label=f"μ*={MU_TRUE}")
    if len(samples) > 0:
        ax_mu.axvline(samples[:, 0].mean(), color=color, lw=1.5,
                      linestyle="--", label=f"μ̂={samples[:,0].mean():.3f}")
    ax_mu.set_xlabel("μ")
    ax_mu.set_ylabel("Density")
    ax_mu.legend(fontsize=7)
    ax_mu.grid(True)

    # ── ROW 2 : histogram of sigma ──
    if len(samples) > 0:
        ax_sig.hist(samples[:, 1], bins=25, density=True,
                    color=color, alpha=0.7, edgecolor="#0f0f14")
    ax_sig.axvline(SIGMA_TRUE, color=TRUE_CLR, lw=1.5, label=f"σ*={SIGMA_TRUE}")
    if len(samples) > 0:
        ax_sig.axvline(samples[:, 1].mean(), color=color, lw=1.5,
                       linestyle="--", label=f"σ̂={samples[:,1].mean():.3f}")
    ax_sig.set_xlabel("σ")
    ax_sig.set_ylabel("Density")
    ax_sig.legend(fontsize=7)
    ax_sig.grid(True)

fig1.tight_layout(rect=[0, 0, 1, 0.97])
fig1.savefig("fig1_prior_impact.png", dpi=150, bbox_inches="tight", facecolor=fig1.get_facecolor())
print("Saved: fig1_prior_impact.png")


# ═══════════════════════════════════════════════════════════════
#  FIGURE 2 — Impact of epsilon   [prior fixed: s=t=1]
# ═══════════════════════════════════════════════════════════════
S_FIXED = 1.0
T_FIXED = 1.0
EPSILONS = [0.1, 0.3, 0.5, 1.0, 2.0]

results_eps = []
for eps in EPSILONS:
    rng = np.random.default_rng(SEED + 100)
    samp, rate = reject_abc(
        Y_obs, L_TRUE, S_FIXED, T_FIXED,
        eps, num_samples=N_SAMPLES,
        max_attempts=500_000, rng=rng
    )
    results_eps.append({
        "eps":      eps,
        "samples":  samp,
        "rate":     rate,
        "mu_hat":   samp[:, 0].mean() if len(samp) > 0 else np.nan,
        "sigma_hat": samp[:, 1].mean() if len(samp) > 0 else np.nan,
    })
    print(f"ε={eps:.2f}  acc={rate:.4%}  μ̂={results_eps[-1]['mu_hat']:.4f}  σ̂={results_eps[-1]['sigma_hat']:.4f}")

n_eps = len(EPSILONS)
cmap  = plt.cm.plasma
eps_colors = [cmap(i / (n_eps - 1)) for i in range(n_eps)]

fig2 = plt.figure(figsize=(16, 9))
fig2.suptitle(
    f"Reject-ABC — Impact of ε   [s=t={S_FIXED}  |  μ* = {MU_TRUE}  σ* = {SIGMA_TRUE}]",
    fontsize=13, color="#e0e0f5"
)

gs = fig2.add_gridspec(2, n_eps + 1, hspace=0.38, wspace=0.35,
                        width_ratios=[1.4] + [1] * n_eps)

# ── left column: 3 summary curves (acceptance rate, mu_hat, sigma_hat) ──
ax_rate  = fig2.add_subplot(gs[0, 0])
ax_est   = fig2.add_subplot(gs[1, 0])

eps_vals  = [r["eps"]      for r in results_eps]
rates     = [r["rate"]     for r in results_eps]
mu_hats   = [r["mu_hat"]   for r in results_eps]
sig_hats  = [r["sigma_hat"] for r in results_eps]

# acceptance rate
ax_rate.plot(eps_vals, rates, "o-", color="#7eb8f7", lw=2, ms=7)
for i, (e, r) in enumerate(zip(eps_vals, rates)):
    ax_rate.annotate(f"{r:.2%}", (e, r), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=7, color=eps_colors[i])
ax_rate.set_title("Acceptance rate vs ε")
ax_rate.set_xlabel("ε")
ax_rate.set_ylabel("Acceptance rate")
ax_rate.grid(True)

# mu_hat and sigma_hat
ax_est.plot(eps_vals, mu_hats,   "o-", color=ACCENT[0], lw=2, ms=7, label="μ̂")
ax_est.plot(eps_vals, sig_hats,  "s-", color=ACCENT[2], lw=2, ms=7, label="σ̂")
ax_est.axhline(MU_TRUE,    color=TRUE_CLR, lw=1.2, linestyle="--", alpha=0.7, label=f"μ*={MU_TRUE}")
ax_est.axhline(SIGMA_TRUE, color=TRUE_CLR, lw=1.2, linestyle=":",  alpha=0.7, label=f"σ*={SIGMA_TRUE}")
ax_est.set_title("Posterior mean vs ε")
ax_est.set_xlabel("ε")
ax_est.set_ylabel("Estimated value")
ax_est.legend(fontsize=7)
ax_est.grid(True)

# ── right columns: one scatter per epsilon ──
for i, res in enumerate(results_eps):
    ax_sc = fig2.add_subplot(gs[0, i + 1])   # top row
    ax_h  = fig2.add_subplot(gs[1, i + 1])   # bottom row

    color = eps_colors[i]
    samp  = res["samples"]

    # scatter
    if len(samp) > 0:
        ax_sc.scatter(samp[:, 0], samp[:, 1],
                      alpha=0.35, s=10, color=color, rasterized=True)
        plot_ellipse(ax_sc, samp, color)
    ax_sc.axvline(MU_TRUE,    color=TRUE_CLR, lw=1.2, linestyle="-")
    ax_sc.axhline(SIGMA_TRUE, color=TRUE_CLR, lw=1.2, linestyle="--")
    ax_sc.set_title(f"ε = {res['eps']}\nacc={res['rate']:.3%}", color=color, fontsize=9)
    ax_sc.set_xlabel("μ", fontsize=8)
    ax_sc.set_ylabel("σ", fontsize=8)
    ax_sc.grid(True)

    # histogram of mu only (bottom row, to keep it readable)
    if len(samp) > 0:
        ax_h.hist(samp[:, 0], bins=20, density=True,
                  color=color, alpha=0.7, edgecolor="#0f0f14")
    ax_h.axvline(MU_TRUE, color=TRUE_CLR, lw=1.4, label=f"μ*")
    if len(samp) > 0:
        ax_h.axvline(samp[:, 0].mean(), color=color, lw=1.4,
                     linestyle="--", label=f"μ̂={samp[:,0].mean():.3f}")
    ax_h.set_xlabel("μ", fontsize=8)
    ax_h.set_ylabel("Density", fontsize=8)
    ax_h.legend(fontsize=6)
    ax_h.grid(True)

fig2.savefig("fig2_epsilon_impact.png", dpi=150, bbox_inches="tight", facecolor=fig2.get_facecolor())
print("Saved: fig2_epsilon_impact.png")

plt.show()


c:\Users\33783\OneDrive - GENES\Documents\Travail\ENSAE\2A\S2\6.Simulation & Monte Carlo\Projet MC\Likelihood-free-inference-and-sums-of-log-normal-variates\src\q1.py:32: RuntimeWarning: overflow encountered in exp
  Y = np.sum(np.exp(X), axis=1)


# Question 2 

In [ ]:
seed_value = 42
main_rng = np.random.default_rng(seed_value)
    
L_true = 10
mu_true = 0.0
sigma_true = 0.3
n_obs = 200  
    
print("Generating strictly defined observational data...")
Y_obs = simulate_lognormal_sum(n_obs, L_true, mu_true, sigma_true, rng=main_rng)

##### epsilon and the prior affect the result ###
    
epsilon_val = 0.5 
step_mu=0.1
step_log_sigma=0.1
    
prior_configs = [
        {"s": 1.0, "t": 1.0, "name": "Well-calibrated"},
        {"s": 10.0, "t": 10.0, "name": "Highly diffuse (Efficiency drop)"},
        {"s": 0.01, "t": 0.01, "name": "Highly concentrated (Prior bias risk)"}
    ]
    
print(f"\n--- MCMC-ABC Empirical Results (Target: mu={mu_true}, sigma={sigma_true}) ---")
for config in prior_configs:
    print(f"\nEvaluating '{config['name']}' prior specification (s={config['s']}, t={config['t']}):")
        
        # Pass the isolated generator down the execution pipeline
    samples, acc_rate = ABCMCMC(
        Y_obs, L_true, config['s'], config['t'], epsilon_val,step_mu,step_log_sigma, num_samples=50, rng=main_rng
        )
        
    print(f"  -> Acceptance Rate : {acc_rate:.4%}")
        
    if len(samples) > 0:
        mu_est = np.mean(samples[:, 0])
        sigma_est = np.mean(samples[:, 1])
        print(f"  -> Estimated mu    : {mu_est:.4f}")
        print(f"  -> Estimated sigma : {sigma_est:.4f}")
    else:
        print("  -> Estimated mu    : N/A (0 samples accepted)")
        print("  -> Estimated sigma : N/A (0 samples accepted)")

Generating strictly defined observational data...

--- MCMC-ABC Empirical Results (Target: mu=0.0, sigma=0.3) ---

Evaluating 'Well-calibrated' prior specification (s=1.0, t=1.0):
  -> Acceptance Rate : 26.0000%
  -> Estimated mu    : -0.0122
  -> Estimated sigma : -2.5856

Evaluating 'Highly diffuse (Efficiency drop)' prior specification (s=10.0, t=10.0):
  -> Acceptance Rate : 20.0000%
  -> Estimated mu    : 0.0256
  -> Estimated sigma : -3.7162

Evaluating 'Highly concentrated (Prior bias risk)' prior specification (s=0.01, t=0.01):
